# Baile con tu foto (MimicMotion + ComfyUI)

Toma **una foto tuya** y **un video de alguien bailando**, y genera un video nuevo donde *tú* haces el baile
(tu cuerpo y tu ropa se mueven como el bailarín). Corre en la GPU gratis de Colab.

**Antes de empezar:** menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU** → Guardar.

Después corre las celdas en orden con ▶:

| Celda | Qué hace | Tiempo |
|---|---|---|
| 1 | Instala ComfyUI, MimicMotion y baja los modelos (~7,5 GB) | 5–10 min (la primera vez) |
| 2 | Sube tu foto y el video del baile | lo que tarde la subida |
| 3 | Genera el video | ~10–25 min para 5 s en una T4 (estimado) |
| 4 | *(Opcional)* Abre la interfaz de ComfyUI para experimentar | 1 min |

**Para que salga bien**
- **Foto:** cuerpo entero (o al menos hasta las rodillas), de pie, de frente, brazos visibles, fondo simple.
  Encuádrala igual que el video (vertical si el video es vertical): la foto se recorta al centro.
- **Video:** una sola persona, cuerpo entero visible todo el tiempo, cámara quieta. Los giros de espaldas y
  los movimientos muy rápidos salen peor.
- La cara suele quedar menos fiel que el cuerpo. Truco: pasa el resultado por **TransferIa** con tu foto como
  `src` para recuperar la cara (y el maquillaje).
- Si la T4 se queda sin memoria: usa un tamaño "rápido" o menos segundos.

**Privacidad:** tu foto y el video se suben a la máquina de Google mientras dura la sesión (se borran al
cerrarla, salvo lo que guardes en tu Drive). Usa solo fotos tuyas o de personas que te dieron permiso, y si
publicas el resultado, indica que está hecho con IA.


In [ ]:
#@title 1. Instalar ComfyUI + MimicMotion y bajar los modelos
#@markdown Marca esto para guardar los modelos en tu Google Drive (~7,5 GB) y no bajarlos cada vez:
guardar_en_drive = False #@param {type:"boolean"}

import os, sys, json, time, uuid, subprocess, urllib.request, urllib.error

RAIZ = os.environ.get("COMFY_RAIZ", "/content/ComfyUI")
PUERTO = 8188
URL = f"http://127.0.0.1:{PUERTO}"
LOG = os.path.join(os.path.dirname(RAIZ), "comfyui.log")
MODELO = "MimicMotionMergedUnet_1-0-fp16.safetensors"


def sh(cmd):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.returncode:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError(f"Falló: {cmd}")
    return r.stdout


def _pedir(ruta, datos=None):
    req = urllib.request.Request(URL + ruta, data=None if datos is None else json.dumps(datos).encode(),
                                 headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            return json.loads(r.read() or b"{}")
    except urllib.error.HTTPError as e:
        return {"_http_error": e.code, **json.loads(e.read() or b"{}")}


def ultimas_lineas_log(n=40):
    if not os.path.exists(LOG):
        return ""
    with open(LOG, encoding="utf-8", errors="replace") as f:
        texto = f.read().replace("\r", "\n")
    return "\n".join([l for l in texto.split("\n") if l.strip()][-n:])


_servidor = None


def iniciar_servidor(extra=()):
    """Arranca ComfyUI en segundo plano (si no está corriendo) y espera a que responda."""
    global _servidor
    try:
        _pedir("/system_stats")
        return
    except Exception:
        pass
    _servidor = subprocess.Popen([sys.executable, "main.py", "--listen", "127.0.0.1", "--port", str(PUERTO), *extra],
                                 cwd=RAIZ, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
    print("Arrancando ComfyUI...", end="", flush=True)
    for _ in range(300):
        if _servidor.poll() is not None:
            print("\n" + ultimas_lineas_log())
            raise RuntimeError("ComfyUI se cerró al arrancar (arriba está el final del registro)")
        try:
            _pedir("/system_stats")
            print(" listo")
            return
        except Exception:
            time.sleep(1)
    raise RuntimeError("ComfyUI no respondió en 5 minutos:\n" + ultimas_lineas_log())


def armar_flujo(foto, video_pose, pasos, semilla, fps, prefijo, modelo=MODELO, ventana=16, solape=4):
    """Flujo de MimicMotion en formato API. Las entradas que no se dan toman el valor por defecto que
    informa el propio ComfyUI (/object_info), así no se rompe si un nodo agrega opciones nuevas."""
    info = _pedir("/object_info")

    def nodo(clase, **valores):
        if clase not in info:
            raise RuntimeError(f"Falta el nodo {clase}: ¿corriste la celda 1 completa?")
        spec = info[clase]["input"]
        entradas = {}
        for nombre, s in spec.get("required", {}).items():
            if nombre in valores:
                entradas[nombre] = valores.pop(nombre)
                continue
            tipo, opciones = s[0], (s[1] if len(s) > 1 and isinstance(s[1], dict) else {})
            if "default" in opciones:
                entradas[nombre] = opciones["default"]
            elif isinstance(tipo, list) and tipo:
                entradas[nombre] = tipo[0]
            elif tipo == "COMBO" and opciones.get("options"):
                entradas[nombre] = opciones["options"][0]
            else:
                raise RuntimeError(f"{clase}: no sé qué poner en '{nombre}'")
        for nombre in list(valores):
            if nombre in spec.get("optional", {}):
                entradas[nombre] = valores.pop(nombre)
        if valores:
            raise RuntimeError(f"{clase} no tiene las entradas {list(valores)}")
        return {"class_type": clase, "inputs": entradas}

    return {
        "foto": nodo("LoadImage", image=foto),
        "video": nodo("VHS_LoadVideoPath", video=video_pose),
        "poses": nodo("MimicMotionGetPoses", ref_image=["foto", 0], pose_images=["video", 0],
                      include_body=True, include_hand=True, include_face=True),
        "modelo": nodo("DownloadAndLoadMimicMotionModel", model=modelo, precision="fp16"),
        # pose_images recibe la salida 0 (la pose de la foto + las del video): asi lo hace el ejemplo del autor
        "generar": nodo("MimicMotionSampler", mimic_pipeline=["modelo", 0], ref_image=["foto", 0],
                        pose_images=["poses", 0], steps=pasos, cfg_min=2.0, cfg_max=2.0, seed=semilla, fps=fps,
                        noise_aug_strength=0.0, context_size=ventana, context_overlap=solape,
                        keep_model_loaded=True),
        "decodificar": nodo("MimicMotionDecode", mimic_pipeline=["modelo", 0], samples=["generar", 0],
                            decode_chunk_size=8),
        "guardar": nodo("SaveImage", images=["decodificar", 0], filename_prefix=prefijo),
    }


def uso_gpu():
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total",
                            "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=10)
        uso, usada, total = [x.strip() for x in r.stdout.split(",")]
        return f"GPU {uso}% · VRAM {int(usada) / 1024:.1f}/{int(total) / 1024:.0f} GB"
    except Exception:
        return ""


def ejecutar(flujo, cada=30):
    """Manda el flujo a ComfyUI y espera. Devuelve True si terminó bien."""
    r = _pedir("/prompt", {"prompt": flujo, "client_id": uuid.uuid4().hex})
    if "prompt_id" not in r:
        print("ComfyUI rechazó el flujo:")
        print(json.dumps(r, indent=1, ensure_ascii=False)[:4000])
        return False
    pid, inicio, ultimo = r["prompt_id"], time.time(), 0
    while True:
        h = _pedir(f"/history/{pid}").get(pid)  # aparece en el historial recién cuando termina
        if h:
            estado = h.get("status", {})
            if estado.get("status_str") == "success":
                print(f"Listo en {(time.time() - inicio) / 60:.1f} min")
                return True
            for tipo, datos in estado.get("messages", []):
                if tipo == "execution_error":
                    print(f"Error en el nodo '{datos.get('node_id')}' ({datos.get('node_type')}): "
                          f"{datos.get('exception_message')}")
                    print("".join(datos.get("traceback", []))[-3000:])
            return False
        if time.time() - ultimo >= cada:
            ultimo = time.time()
            linea = (ultimas_lineas_log(1) or "").strip()[-140:]
            print(f"[{(time.time() - inicio) / 60:4.1f} min] {uso_gpu()} | {linea}", flush=True)
        time.sleep(3)


import shutil
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                   capture_output=True, text=True)
if r.returncode:
    raise SystemExit("No hay GPU. Menú Entorno de ejecución > Cambiar tipo de entorno de ejecución > T4 GPU, "
                     "guarda y vuelve a correr esta celda.")
print("GPU:", r.stdout.strip())

NODOS = {
    "ComfyUI-MimicMotionWrapper": "https://github.com/kijai/ComfyUI-MimicMotionWrapper",
    "ComfyUI-VideoHelperSuite": "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite",
    "ComfyUI-KJNodes": "https://github.com/kijai/ComfyUI-KJNodes",  # solo lo usa el ejemplo de la celda 4
}
if not os.path.isdir(RAIZ):
    print("Bajando ComfyUI...")
    sh(f"git clone -q --depth 1 https://github.com/comfyanonymous/ComfyUI {RAIZ}")
for nombre, repo in NODOS.items():
    if not os.path.isdir(f"{RAIZ}/custom_nodes/{nombre}"):
        sh(f"git clone -q --depth 1 {repo} {RAIZ}/custom_nodes/{nombre}")
print("Instalando dependencias...")
sh(f"pip install -q -r {RAIZ}/requirements.txt")
for nombre in NODOS:
    sh(f"pip install -q -r {RAIZ}/custom_nodes/{nombre}/requirements.txt")
# Colab trae un torchao que no calza con su diffusers ("cannot import name 'FqnToConfig'") y diffusers lo
# importa si esta instalado. MimicMotion no lo usa: se quita.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], capture_output=True)
r = subprocess.run([sys.executable, "-c", "from diffusers.models import AutoencoderKLTemporalDecoder"],
                   capture_output=True, text=True)
if r.returncode:
    print(r.stderr[-3000:])
    raise RuntimeError("La librería diffusers no se puede importar (arriba está el error)")

# Carpetas de modelos (en el Drive si se pidio: se enlazan para que ComfyUI las vea en su lugar)
CARPETAS = {
    "mimicmotion": f"{RAIZ}/models/mimicmotion",
    "diffusers": f"{RAIZ}/models/diffusers",
    "DWPose": f"{RAIZ}/custom_nodes/ComfyUI-MimicMotionWrapper/models/DWPose",
}
if guardar_en_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    for nombre, ruta in CARPETAS.items():
        en_drive = f"/content/drive/MyDrive/ComfyUI_modelos/{nombre}"
        os.makedirs(en_drive, exist_ok=True)
        if os.path.isdir(ruta) and not os.path.islink(ruta):
            for f in os.listdir(ruta):
                shutil.move(os.path.join(ruta, f), en_drive)
            os.rmdir(ruta)
        if not os.path.islink(ruta):
            os.makedirs(os.path.dirname(ruta), exist_ok=True)
            os.symlink(en_drive, ruta)

# Se bajan aqui (con barra de progreso) y no dentro de ComfyUI: mismos archivos y carpetas que usa el nodo
from huggingface_hub import snapshot_download
print("Bajando modelos (la primera vez tarda)...")
snapshot_download(repo_id="Kijai/MimicMotion_pruned", allow_patterns=[f"*{MODELO}*"],
                  local_dir=CARPETAS["mimicmotion"])
snapshot_download(repo_id="vdo/stable-video-diffusion-img2vid-xt-1-1", allow_patterns=["*.json", "*fp16*"],
                  ignore_patterns=["*unet*"],
                  local_dir=f"{CARPETAS['diffusers']}/stable-video-diffusion-img2vid-xt-1-1")
snapshot_download(repo_id="hr16/yolox-onnx", allow_patterns=["*yolox_l.torchscript.pt*"],
                  local_dir=CARPETAS["DWPose"])
snapshot_download(repo_id="hr16/DWPose-TorchScript-BatchSize5",
                  allow_patterns=["*dw-ll_ucoco_384_bs5.torchscript.pt*"], local_dir=CARPETAS["DWPose"])

# Si ya habia un ComfyUI corriendo (celda corrida otra vez), se reinicia para que cargue los nodos de nuevo
subprocess.run(["pkill", "-f", f"main.py --listen 127.0.0.1 --port {PUERTO}"])
time.sleep(3)
iniciar_servidor()
faltan = [c for c in ("MimicMotionSampler", "VHS_LoadVideoPath") if c not in _pedir("/object_info")]
if faltan:
    print(ultimas_lineas_log(60))
    raise RuntimeError(f"ComfyUI arrancó pero no cargó {faltan} (arriba está el registro)")
print("✅ Todo instalado. Sigue con la celda 2.")


In [ ]:
#@title 2. Subir tu foto y el video del baile
from google.colab import files

ENTRADA = "/content/entrada"
os.makedirs(ENTRADA, exist_ok=True)

def subir(que):
    print(f"Elige {que}:")
    subido = files.upload()
    if not subido:
        raise SystemExit("No se subió nada")
    nombre, datos = next(iter(subido.items()))
    ruta = os.path.join(ENTRADA, nombre)
    with open(ruta, "wb") as f:
        f.write(datos)
    os.remove(nombre)  # files.upload tambien lo deja en /content
    print(f"  -> {nombre} ({len(datos) / 1e6:.1f} MB)")
    return ruta

FOTO = subir("TU FOTO (jpg/png, cuerpo entero)")
VIDEO = subir("EL VIDEO DEL BAILE (mp4/mov)")
print(sh(f'ffprobe -v error -select_streams v:0 -show_entries stream=width,height,r_frame_rate:format=duration '
         f'-of default=nw=1 "{VIDEO}"'))
print("✅ Sigue con la celda 3.")


In [ ]:
#@title 3. Generar el video
#@markdown **Qué parte del video del baile** (en segundos). Empieza con 3–5 s para probar.
inicio_seg = 0 #@param {type:"number"}
duracion_seg = 5 #@param {type:"number"}
#@markdown **Tamaño:** usa el mismo sentido que el video. Los "rápidos" gastan menos memoria y tiempo.
tamano = "vertical 576x1024" #@param ["vertical 576x1024", "vertical rápido 448x768", "horizontal 1024x576", "horizontal rápido 768x448"]
#@markdown **Pasos:** más = un poco mejor y más lento. **Semilla:** cambia el número para otra variante.
pasos = 20 #@param {type:"slider", min:10, max:40, step:1}
semilla = 42 #@param {type:"integer"}
fps = 15 #@param {type:"integer"}
#@markdown Guardar también un video comparando el baile original y el tuyo, uno al lado del otro:
lado_a_lado = True #@param {type:"boolean"}
#@markdown ---
#@markdown **Velocidad.** El video se genera por *ventanas* de cuadros que se solapan; lo solapado se calcula
#@markdown dos veces. Ventanas más grandes = menos trabajo repetido (más rápido) pero más VRAM.
#@markdown - **1.0, ventana 16, solape 4**: lo seguro (~6 GB de VRAM).
#@markdown - **1.0, ventana 24 o 32**: ~10–20 % más rápido, usa más VRAM; el modelo 1.0 se entrenó con 16,
#@markdown   así que mira si el movimiento sale igual de bien.
#@markdown - **1.1**: versión más nueva del modelo, se entrenó con ventanas de 72 (la ventana elegida se
#@markdown   ignora). ~20–30 % más rápido y suele verse mejor, pero puede no entrar en la T4 (más aún en tamaño
#@markdown   normal) y necesita al menos ~5 s de video. La primera vez baja 3 GB más.
modelo_mimic = "1.0" #@param ["1.0", "1.1"]
ventana = 16 #@param [16, 24, 32] {type:"raw"}
solape = 4 #@param {type:"slider", min:2, max:8, step:1}

import glob, datetime
from base64 import b64encode
from PIL import Image, ImageOps
from IPython.display import HTML, display

if "FOTO" not in globals():
    raise SystemExit("Primero corre la celda 2")
W, H = map(int, tamano.split()[-1].split("x"))
cuadros = int(round(duracion_seg * fps))
if modelo_mimic == "1.1":
    archivo_modelo, ventana = "MimicMotionMergedUnet_1-1-fp16.safetensors", 72
else:
    archivo_modelo = MODELO
if solape >= ventana:
    raise SystemExit("El solape tiene que ser menor que la ventana")
if cuadros + 1 < ventana:
    raise SystemExit(f"Muy corto: con ventana {ventana} hacen falta al menos {ventana} cuadros "
                     f"({(ventana - 1) / fps:.1f} s a {fps} fps). Sube la duración o usa una ventana menor.")
if not os.path.exists(f"{RAIZ}/models/mimicmotion/{archivo_modelo}"):
    from huggingface_hub import snapshot_download
    print(f"Bajando {archivo_modelo} (3 GB, solo la primera vez)...")
    snapshot_download(repo_id="Kijai/MimicMotion_pruned", allow_patterns=[f"*{archivo_modelo}*"],
                      local_dir=f"{RAIZ}/models/mimicmotion")

TRABAJO = "/content/trabajo"
os.makedirs(TRABAJO, exist_ok=True)
pose = f"{TRABAJO}/pose.mp4"
audio = f"{TRABAJO}/audio.m4a"
sh(f'ffmpeg -y -loglevel error -ss {inicio_seg} -t {duracion_seg} -i "{VIDEO}" '
   f'-vf "fps={fps},scale={W}:{H}:force_original_aspect_ratio=increase,crop={W}:{H},setsar=1" '
   f'-an -c:v libx264 -crf 10 -pix_fmt yuv420p "{pose}"')
if os.path.exists(audio):
    os.remove(audio)
tiene_audio = subprocess.run(f'ffmpeg -y -loglevel error -ss {inicio_seg} -t {duracion_seg} -i "{VIDEO}" '
                             f'-vn -c:a aac "{audio}"', shell=True).returncode == 0 and os.path.exists(audio)

foto = ImageOps.exif_transpose(Image.open(FOTO)).convert("RGB")
if abs(foto.width / foto.height - W / H) > 0.25:
    print(f"⚠️ La foto ({foto.width}x{foto.height}) tiene otra forma que {W}x{H}: se recorta al centro y "
          f"puede quedar parte del cuerpo afuera. Mejor una foto encuadrada como el video.")
ImageOps.fit(foto, (W, H), Image.LANCZOS).save(f"{RAIZ}/input/foto_baile.png")

iniciar_servidor()
carpeta = "baile_" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
flujo = armar_flujo("foto_baile.png", pose, pasos, semilla, fps, f"{carpeta}/f", modelo=archivo_modelo,
                    ventana=ventana, solape=solape)
print(f"Generando {cuadros} cuadros de {W}x{H} ({duracion_seg} s a {fps} fps), modelo {modelo_mimic}, "
      f"ventana {ventana}, solape {solape}...")
if not ejecutar(flujo):
    raise SystemExit("Falló la generación (el motivo está arriba). Si dice 'out of memory', "
                     "usa un tamaño rápido, menos segundos, una ventana menor o el modelo 1.0.")

png = sorted(glob.glob(f"{RAIZ}/output/{carpeta}/*.png"))
SALIDA = "/content/resultados"
os.makedirs(SALIDA, exist_ok=True)
resultado = f"{SALIDA}/{carpeta}.mp4"
con_audio = f'-i "{audio}" -map 0:v -map 1:a -c:a copy -shortest' if tiene_audio else ""
sh(f'ffmpeg -y -loglevel error -framerate {fps} -i "{RAIZ}/output/{carpeta}/f_%05d_.png" '
   f'{con_audio} -c:v libx264 -crf 17 -pix_fmt yuv420p "{resultado}"')
videos = [resultado]
if lado_a_lado:
    comparacion = f"{SALIDA}/{carpeta}_comparacion.mp4"
    entrada_audio, mapa_audio = (f'-i "{audio}"', "-map 2:a -c:a copy -shortest") if tiene_audio else ("", "")
    sh(f'ffmpeg -y -loglevel error -i "{pose}" -i "{resultado}" {entrada_audio} '
       f'-filter_complex "[0:v][1:v]hstack=inputs=2[v]" -map "[v]" {mapa_audio} '
       f'-c:v libx264 -crf 17 -pix_fmt yuv420p "{comparacion}"')
    videos.append(comparacion)

if globals().get("guardar_en_drive"):
    os.makedirs("/content/drive/MyDrive/Baile_resultados", exist_ok=True)
    for v in videos:
        shutil.copy(v, "/content/drive/MyDrive/Baile_resultados")
    print("Copiado también a tu Drive: MyDrive/Baile_resultados")

print(f"✅ {len(png)} cuadros -> {', '.join(os.path.basename(v) for v in videos)}")
datos = b64encode(open(videos[-1], "rb").read()).decode()
display(HTML(f'<video controls loop autoplay muted style="max-height:600px" '
             f'src="data:video/mp4;base64,{datos}"></video>'))
for v in videos:
    files.download(v)


In [ ]:
#@title 4. (Opcional) Abrir la interfaz de ComfyUI
#@markdown Abre ComfyUI en otra pestaña para experimentar. Trae cargado el flujo de ejemplo del autor de
#@markdown MimicMotion (menú Flujos de trabajo → "MimicMotion ejemplo"): ahí eliges tu foto y tu video en los
#@markdown nodos "Load Image" y "Load Video". Deja esta celda corriendo mientras la usas.
import re, threading

iniciar_servidor()
flujos = f"{RAIZ}/user/default/workflows"
os.makedirs(flujos, exist_ok=True)
shutil.copy(f"{RAIZ}/custom_nodes/ComfyUI-MimicMotionWrapper/examples/mimic_motion_example_02.json",
            f"{flujos}/MimicMotion ejemplo.json")

if not os.path.exists("/content/cloudflared"):
    sh("wget -q -O /content/cloudflared "
       "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
       "&& chmod +x /content/cloudflared")
tunel = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", URL],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
direccion = None
for linea in tunel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", linea)
    if m:
        direccion = m.group(0)
        break
threading.Thread(target=lambda: [None for _ in tunel.stdout], daemon=True).start()  # vacia la salida
if not direccion:
    raise RuntimeError("No se pudo abrir el túnel; vuelve a correr la celda")
print("Abre ComfyUI aquí (tarda unos segundos en responder la primera vez):", direccion)
display(HTML(f'<a href="{direccion}" target="_blank" style="font-size:18px">{direccion}</a>'))
tunel.wait()


## Si algo falla

- **"No hay GPU"**: cambia el entorno a T4 GPU (menú *Entorno de ejecución*) y corre otra vez desde la celda 1.
- **"out of memory" / CUDA**: tamaño "rápido", menos segundos, o *Entorno de ejecución → Reiniciar sesión* y
  vuelve a correr 1 → 3.
- **Error en `MimicMotionGetPoses`**: no encontró un cuerpo en la foto o en el video. Usa una foto de cuerpo
  entero y un video donde se vea la persona completa.
- **Se desconectó**: Colab gratis corta sesiones largas o inactivas. Con *guardar_en_drive* los modelos no se
  vuelven a bajar.

Créditos: [MimicMotion](https://github.com/Tencent/MimicMotion) (Tencent),
[ComfyUI-MimicMotionWrapper](https://github.com/kijai/ComfyUI-MimicMotionWrapper) (kijai),
[ComfyUI](https://github.com/comfyanonymous/ComfyUI). Revisa sus licencias antes de usarlo con fines comerciales.
